In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
#basemessage is collection of all HumanMessage,AIMessage
from typing import TypedDict, Annotated

from langgraph.checkpoint.memory import MemorySaver #now we are using RAM to store the chat history ***
 

c:\Users\Sachin S\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
       messages:Annotated[list[BaseMessage], add_messages]

In [3]:
model=ChatOpenAI()

def chat_node(state:ChatState):
    #take user query from state
    messages=state['messages']
    #send to llm
    response=model.invoke(messages)
    #response store state
    return {'messages':[response]}

In [4]:
checkpointer=MemorySaver()  #*****

graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile(checkpointer=checkpointer)  #****
inital_state={
    'messages':[HumanMessage(content='What is the capital of India')]
}

chatbot.invoke(inital_state)




ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [6]:
thread_id=1  #**** thread id for each conversation chat session

while True:
    user_message=input('TypeHere:')
    print("User:",user_message)

    if user_message.strip().lower() in ['quit','exit','bye']:
        break

    config={'configurable':{'thread_id':thread_id}} # ***

    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print("AI Message: ",response['messages'][-1].content)

'''Here the problem with the chatbot isis resolved where the  chat history will be maintained'''


User: Hi My name is Sachin
AI Message:  Hello Sachin! It's nice to meet you. How can I assist you today?
User: WHat is my  name
AI Message:  Your name is Sachin.
User: can you add 2 and 3
AI Message:  Of course! 2 + 3 equals 5. Let me know if there's anything else you'd like me to help you with.
User: now can you multiply the result with 6
AI Message:  Sure! 

5 (result of 2 + 3) multiplied by 6 is 30.
User: exit


'Here the problem with the chatbot isis resolved where the  chat history will be maintained'

In [9]:
chatbot.get_state(config=config)
# every time the chath history will be appended and stores in the RAM and the LLM will read all the  chat history and LLM will respond 

StateSnapshot(values={'messages': [HumanMessage(content='Hi ', additional_kwargs={}, response_metadata={}, id='0a8dd59f-c8e2-4880-850c-25e88c8fc19c'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EFjw6MJSVjB3fYMvXlqti2SDs3sTA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02a8b-df1d-79a0-9470-f8d9a0eaca03-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_detail